# Data & Tools Exploration

This notebook walks you through the foundational layer of the AML investigation setup:
- What the database looks like and how it's built
- How to explore transactions and accounts manually
- What the `ReadOnlySqlDatabase` tool is and why it exists
- How an agent would "see" the database through that tool

In [1]:
import os
import sqlite3
from pathlib import Path

import pandas as pd
from aieng.agent_evals.aml_investigation.data import download_dataset_file, normalize_transactions_data
from aieng.agent_evals.tools.sql_database import ReadOnlySqlDatabase
from dotenv import load_dotenv


# Setting the notebook directory to the project's root folder
if Path("").absolute().name == "eval-agents":
    print(f"Notebook path is already the root path: {Path('').absolute()}")
else:
    os.chdir(Path("").absolute().parent.parent)
    print(f"The notebook path has been set to: {Path('').absolute()}")

load_dotenv(verbose=True)

The notebook path has been set to: /home/coder/eval-agents


True

## 1. Exploring the dataset

We will be using the [IBM Transactions for Anti Money Laundering (AML)](https://www.kaggle.com/datasets/ealtman2019/ibm-transactions-for-anti-money-laundering-aml) dataset, which is available on Kaggle. It contains synthetic transaction data designed to mimic real-world financial transactions, including both legitimate and potentially fraudulent activities. The dataset includes various features such as transaction amount, type, origin and destination accounts, timestamps, and a label indicating whether the transaction is fraudulent or not.

### 1.1 Downloading the dataset

There are 6 datasets available, divided into two groups of three sets. The groups are based on the ratio of illicit transactions in the data:
- Group **HI** contains relatively higher illicit transaction ratios (i.e. more laundering activity)
- Group **LI** contains relatively lower illicit transaction ratios (i.e. less laundering activity)

Each group has three sets of data based on the total number of transactions/accounts: "Small", "Medium", and "Large".

You can download any of the six datasets using the `download_dataset_file` function. However, **note that the code tries to load all the data into memory at once, so the "Medium" and "Large" datasets may cause memory issues on some machines**. For this reason, we recommend starting with the "Small" dataset.

Each dataset has 3 files that you can download separately:
- `<HI/LI>_<Small/Medium/Large>_Trans.csv`: contains transaction data, with each row representing a single transaction.
- `<HI/LI>_<Small/Medium/Large>_accounts.csv`: contains account data, with each row representing a single account.
- `<HI/LI>_<Small/Medium/Large>_Patterns.txt`: contains ground-truth laundering patterns, which are groups of transactions that are known to be part of the same laundering scheme. Each pattern includes a list of transaction IDs that are involved in that pattern.

#### 1.1.1. The transactions

In [2]:
path_to_transactions_csv = download_dataset_file(illicit_ratio="HI", transactions_size="Small", filename="Trans.csv")
print(f"Path to transactions.csv: {path_to_transactions_csv}")

100%|██████████| 454M/454M [00:05<00:00, 81.0MB/s] 

Path to transactions.csv: /home/coder/.cache/kagglehub/datasets/ealtman2019/ibm-transactions-for-anti-money-laundering-aml/versions/8/HI-Small_Trans.csv


In [3]:
transactions_df = pd.read_csv(path_to_transactions_csv)
transactions_df.head(10)

,Timestamp,From Bank,Account,To Bank,Account.1,Amount Received,Receiving Currency,Amount Paid,Payment Currency,Payment Format,Is Laundering
0,2022/09/01 00:20,10,8000EBD30,10,8000EBD30,3697.34,US Dollar,3697.34,US Dollar,Reinvestment,0
1,2022/09/01 00:20,3208,8000F4580,1,8000F5340,0.01,US Dollar,0.01,US Dollar,Cheque,0
2,2022/09/01 00:00,3209,8000F4670,3209,8000F4670,14675.57,US Dollar,14675.57,US Dollar,Reinvestment,0
3,2022/09/01 00:02,12,8000F5030,12,8000F5030,2806.97,US Dollar,2806.97,US Dollar,Reinvestment,0
4,2022/09/01 00:06,10,8000F5200,10,8000F5200,36682.97,US Dollar,36682.97,US Dollar,Reinvestment,0
5,2022/09/01 00:03,1,8000F5AD0,1,8000F5AD0,6162.44,US Dollar,6162.44,US Dollar,Reinvestment,0
6,2022/09/01 00:08,1,8000EBAC0,1,8000EBAC0,14.26,US Dollar,14.26,US Dollar,Reinvestment,0
7,2022/09/01 00:16,1,8000EC1E0,1,8000EC1E0,11.86,US Dollar,11.86,US Dollar,Reinvestment,0
8,2022/09/01 00:26,12,8000EC280,2439,8017BF800,7.66,US Dollar,7.66,US Dollar,Credit Card,0
9,2022/09/01 00:21,1,8000EDEC0,211050,80AEF5310,383.71,US Dollar,383.71,US Dollar,Credit Card,0


In [4]:
# Are there duplicates?
print(f"Number of duplicate transactions: {transactions_df.duplicated().sum()}")

Number of duplicate transactions: 9


Notice that the transactions dataset needs some cleaning. For example:
- There are duplicate transactions that should be removed before analysis.
- There are two columns that have the same name "Account". Pandas automatically renamed the second one to "Account.1", but we should rename them to something more descriptive.

We use the `normalize_transactions_data` function to perform these cleaning steps and make the transactions data easier to work with.

In [5]:
transactions_df = normalize_transactions_data(transactions_df)
transactions_df.head(10)

,timestamp,from_bank,from_account,to_bank,to_account,amount_received,receiving_currency,amount_paid,payment_currency,payment_format,is_laundering,transaction_id
0,2022-09-01T00:00:00,32282,801B64CE0,32282,801B64CE0,29294.03,US Dollar,29294.03,US Dollar,Reinvestment,0,b22960f6bc5dae37
1,2022-09-01T00:00:00,212818,804B1DF50,212818,804B1DF50,538528.53,Rupee,538528.53,Rupee,Reinvestment,0,a4f2e09e46f15d6e
2,2022-09-01T00:00:00,16927,8062913A0,28,806291440,96718657.00,Ruble,96718657.00,Ruble,ACH,0,fbc9ba9abe540e30
3,2022-09-01T00:00:00,131086,80CB3F220,131086,80CB3F220,20289185.18,Mexican Peso,20289185.18,Mexican Peso,Reinvestment,0,8c9b1019003fbd56
4,2022-09-01T00:00:00,1068,802AC9790,17425,80EC2E740,3174.84,US Dollar,3174.84,US Dollar,Cheque,0,5ec710f6f19a95da
5,2022-09-01T00:00:00,13264,803D0EB90,227171,80EC2DDC0,3198.62,US Dollar,3198.62,US Dollar,ACH,0,02c4d40b869bd284
6,2022-09-01T00:00:00,45701,81130A2B0,45701,81130A2B0,6160.74,Shekel,6160.74,Shekel,Reinvestment,0,4f5d7717956140c7
7,2022-09-01T00:00:00,8354,807350F90,8354,807350F90,17.71,US Dollar,17.71,US Dollar,Reinvestment,0,482dad2c990b5202
8,2022-09-01T00:00:00,1132,80BE5E680,1132,80BE5E680,1057246.47,Mexican Peso,1057246.47,Mexican Peso,Reinvestment,0,32936ef0a9a79bc4
9,2022-09-01T00:00:00,39900,803D91A80,210261,803D92040,0.03,Yen,0.03,Yen,Cheque,0,7b310ee1aaf1d6c9


#### 1.1.2. The accounts

In [6]:
path_to_accounts_csv = download_dataset_file(illicit_ratio="HI", transactions_size="Small", filename="accounts.csv")
print(f"Path to accounts.csv: {path_to_accounts_csv}")

100%|██████████| 32.5M/32.5M [00:00<00:00, 61.6MB/s]

Path to accounts.csv: /home/coder/.cache/kagglehub/datasets/ealtman2019/ibm-transactions-for-anti-money-laundering-aml/versions/8/HI-Small_accounts.csv


In [7]:
accounts_df = pd.read_csv(path_to_accounts_csv)
accounts_df.head(10)

,Bank Name,Bank ID,Account Number,Entity ID,Entity Name
0,Portugal Bank #4507,331579,80B779D80,80062E240,Sole Proprietorship #50438
1,Canada Bank #27,210,809D86900,800C998A0,Corporation #33520
2,UK Bank #33,21884,80812BE00,800C47F50,Partnership #35397
3,Germany Bank #4815,32742,81047F300,80096F0B0,Corporation #48813
4,National Bank of Harrisburg,127390,80BD8CF00,800FB8760,Corporation #889
5,Spain Bank #439,224555,80F269580,80064EB20,Sole Proprietorship #42987
6,Savings Bank of Omaha,32013,80E6E8680,800CCEDE0,Sole Proprietorship #20855
7,Brazil Bank #39,335355,80CDEFD80,800D88F10,Partnership #36822
8,Mexico Bank #16,1132,80B723600,800D3F760,Partnership #36511
9,Russia Bank #39,217824,806B17000,800B44480,Partnership #33830


Similar to the transactions dataset, we can rename the columns in the accounts dataset to make them easier to work with.

In [8]:
accounts_df.rename(
    columns={
        "Bank Name": "bank_name",
        "Bank ID": "bank_id",
        "Account Number": "account_number",
        "Entity ID": "entity_id",
        "Entity Name": "entity_name",
    },
    inplace=True,
)
accounts_df.head(10)

,bank_name,bank_id,account_number,entity_id,entity_name
0,Portugal Bank #4507,331579,80B779D80,80062E240,Sole Proprietorship #50438
1,Canada Bank #27,210,809D86900,800C998A0,Corporation #33520
2,UK Bank #33,21884,80812BE00,800C47F50,Partnership #35397
3,Germany Bank #4815,32742,81047F300,80096F0B0,Corporation #48813
4,National Bank of Harrisburg,127390,80BD8CF00,800FB8760,Corporation #889
5,Spain Bank #439,224555,80F269580,80064EB20,Sole Proprietorship #42987
6,Savings Bank of Omaha,32013,80E6E8680,800CCEDE0,Sole Proprietorship #20855
7,Brazil Bank #39,335355,80CDEFD80,800D88F10,Partnership #36822
8,Mexico Bank #16,1132,80B723600,800D3F760,Partnership #36511
9,Russia Bank #39,217824,806B17000,800B44480,Partnership #33830


#### 1.1.3. The patterns

In [9]:
path_to_patterns_txt = download_dataset_file(illicit_ratio="HI", transactions_size="Small", filename="Patterns.txt")
print(f"Path to patterns.txt: {path_to_patterns_txt}")

100%|██████████| 316k/316k [00:00<00:00, 2.51MB/s]

Path to patterns.txt: /home/coder/.cache/kagglehub/datasets/ealtman2019/ibm-transactions-for-anti-money-laundering-aml/versions/8/HI-Small_Patterns.txt


Laundering patterns start with `BEGIN LAUNDERING ATTEMPT` and end with `END LAUNDERING ATTEMPT`. Each pattern includes a list of transactions that are involved in that pattern.

In [10]:
# Print the first laundering pattern
begin_prefix = "BEGIN LAUNDERING ATTEMPT - "
end_prefix = "END LAUNDERING ATTEMPT"
with open(path_to_patterns_txt, "r") as f:
    for line in f:
        if line.startswith(begin_prefix):
            print(f"\n{line.strip()}")
        elif line.startswith(end_prefix):
            print(f"{line.strip()}")
            break
        else:
            print(line.strip())


BEGIN LAUNDERING ATTEMPT - FAN-OUT:  Max 16-degree Fan-Out
2022/09/01 00:06,021174,800737690,012,80011F990,2848.96,Euro,2848.96,Euro,ACH,1
2022/09/01 04:33,021174,800737690,020,80020C5B0,8630.40,Euro,8630.40,Euro,ACH,1
2022/09/01 09:14,021174,800737690,020,80006A5E0,35642.49,Yuan,35642.49,Yuan,ACH,1
2022/09/01 09:56,021174,800737690,00220,8007A5B70,5738987.96,US Dollar,5738987.96,US Dollar,ACH,1
2022/09/01 11:28,021174,800737690,001244,80093C0D0,7254.53,US Dollar,7254.53,US Dollar,ACH,1
2022/09/01 13:13,021174,800737690,00513,80078E200,6990.87,US Dollar,6990.87,US Dollar,ACH,1
2022/09/01 14:11,021174,800737690,020,80066B990,12536.92,Euro,12536.92,Euro,ACH,1
2022/09/02 15:40,021174,800737690,00410,8002CC310,3511.82,Euro,3511.82,Euro,ACH,1
2022/09/02 21:23,021174,800737690,01292,8004030A0,16135.09,US Dollar,16135.09,US Dollar,ACH,1
2022/09/02 23:10,021174,800737690,01601,800578800,12183.28,US Dollar,12183.28,US Dollar,ACH,1
2022/09/03 09:29,021174,800737690,001,800AAF0B0,15197.45,US Dol

## 2. Build the Database

With the datasets downloaded and cleaned, we can build a SQLite database that contains the transactions and accounts data. This will allow us to query the data using SQL, which is a common way for agents to interact with databases.

In [11]:
DB_PATH = Path("implementations/aml_investigation/data/aml_transactions.db")
DDL_PATH = Path("implementations/aml_investigation/data/schema.ddl")

print(f"Database exists: {DB_PATH.exists()}")
print(f"DDL file exists: {DDL_PATH.exists()}")

Database exists: False
DDL file exists: True


In [12]:
# Run this cell only if the database doesn't exist yet.
# It will build the SQLite DB. It may take some time to run.

if not DB_PATH.exists():
    with sqlite3.connect(DB_PATH) as conn:
        conn.execute("PRAGMA foreign_keys = ON;")

        if not DDL_PATH.exists():
            raise FileNotFoundError(f"DDL file not found at {DDL_PATH}")

        with open(DDL_PATH, "r") as file:
            conn.executescript(file.read())
        conn.commit()

        # Add new columns to the transactions DataFrame: date, day_of_week, time_of_day
        transactions_df["date"] = pd.to_datetime(transactions_df["timestamp"]).dt.date
        transactions_df["day_of_week"] = pd.to_datetime(transactions_df["timestamp"]).dt.day_name()
        transactions_df["time_of_day"] = pd.to_datetime(transactions_df["timestamp"]).dt.time

        # Set Transaction ID as index
        transactions_df.set_index("transaction_id", drop=True, inplace=True)

        # NOTE: We drop the "is_laundering" column since that's the label the
        # agent is trying to predict, and it wouldn't be present in a real
        # investigation scenario.
        transactions_df.drop(columns=["is_laundering"], inplace=True)

        accounts_df.to_sql("accounts", conn, if_exists="append", index=False)
        transactions_df.to_sql("transactions", conn, if_exists="append")
else:
    print("Database already exists — skipping creation.")

### 2.1. Understand the Schema

The database has two core tables and one convenience view:

- **`accounts`** — who owns each account (bank, account number, entity name)
- **`transactions`** — every transfer between accounts (amount, currency, timestamp, payment format)
- **`account_transactions`** (view) — a flattened, account-centric view of transactions. Each transaction appears **twice**: once as an OUT row for the sender, once as an IN row for the receiver. This makes it easy to query all activity for a single account without a `UNION` every time.

Let's look at the raw DDL:

In [13]:
print(DDL_PATH.read_text())

DROP VIEW IF EXISTS "account_transactions";

DROP TABLE IF EXISTS "accounts";
CREATE TABLE "accounts" (
    "bank_name" TEXT,
    "bank_id" INTEGER,
    "account_number" TEXT,
    "entity_id" TEXT,
    "entity_name" TEXT,
    PRIMARY KEY ("bank_id", "account_number")
);

DROP TABLE IF EXISTS "transactions";
CREATE TABLE "transactions" (
    "transaction_id" TEXT PRIMARY KEY,
    "timestamp" Text,
    "date" TEXT,
    "day_of_week" TEXT,
    "time_of_day" TEXT,
    "from_bank" INTEGER,
    "from_account" TEXT,
    "to_bank" INTEGER,
    "to_account" TEXT,
    "amount_received" REAL,
    "receiving_currency" TEXT,
    "amount_paid" REAL,
    "payment_currency" TEXT,
    "payment_format" TEXT,
    FOREIGN KEY ("from_bank", "from_account")
        REFERENCES "accounts" ("bank_id", "account_number"),
    FOREIGN KEY ("to_bank", "to_account")
        REFERENCES "accounts" ("bank_id", "account_number")
);

CREATE VIEW account_transactions AS
SELECT
  transaction_id,
  timestamp,
  from_accoun

## 3. Manual Exploration with `pandas` + `sqlite3`

Let's get familiar with the data before involving any agent.

In [14]:
conn = sqlite3.connect(DB_PATH)

# Quick sanity check: how many rows in each table?
for table in ["accounts", "transactions"]:
    count = pd.read_sql(f"SELECT COUNT(*) AS n FROM {table}", conn).iloc[0]["n"]
    print(f"{table:20s}: {count:,} rows")

accounts            : 518,581 rows
transactions        : 5,078,336 rows


In [15]:
# Preview the accounts table
pd.read_sql("SELECT * FROM accounts LIMIT 10", conn)

,bank_name,bank_id,account_number,entity_id,entity_name
0,Portugal Bank #4507,331579,80B779D80,80062E240,Sole Proprietorship #50438
1,Canada Bank #27,210,809D86900,800C998A0,Corporation #33520
2,UK Bank #33,21884,80812BE00,800C47F50,Partnership #35397
3,Germany Bank #4815,32742,81047F300,80096F0B0,Corporation #48813
4,National Bank of Harrisburg,127390,80BD8CF00,800FB8760,Corporation #889
5,Spain Bank #439,224555,80F269580,80064EB20,Sole Proprietorship #42987
6,Savings Bank of Omaha,32013,80E6E8680,800CCEDE0,Sole Proprietorship #20855
7,Brazil Bank #39,335355,80CDEFD80,800D88F10,Partnership #36822
8,Mexico Bank #16,1132,80B723600,800D3F760,Partnership #36511
9,Russia Bank #39,217824,806B17000,800B44480,Partnership #33830


In [16]:
# Preview the transactions table
pd.read_sql("SELECT * FROM transactions LIMIT 10", conn)

,transaction_id,timestamp,date,day_of_week,time_of_day,from_bank,from_account,to_bank,to_account,amount_received,receiving_currency,amount_paid,payment_currency,payment_format
0,b22960f6bc5dae37,2022-09-01T00:00:00,2022-09-01,Thursday,00:00:00.000000,32282,801B64CE0,32282,801B64CE0,29294.03,US Dollar,29294.03,US Dollar,Reinvestment
1,a4f2e09e46f15d6e,2022-09-01T00:00:00,2022-09-01,Thursday,00:00:00.000000,212818,804B1DF50,212818,804B1DF50,538528.53,Rupee,538528.53,Rupee,Reinvestment
2,fbc9ba9abe540e30,2022-09-01T00:00:00,2022-09-01,Thursday,00:00:00.000000,16927,8062913A0,28,806291440,96718657.00,Ruble,96718657.00,Ruble,ACH
3,8c9b1019003fbd56,2022-09-01T00:00:00,2022-09-01,Thursday,00:00:00.000000,131086,80CB3F220,131086,80CB3F220,20289185.18,Mexican Peso,20289185.18,Mexican Peso,Reinvestment
4,5ec710f6f19a95da,2022-09-01T00:00:00,2022-09-01,Thursday,00:00:00.000000,1068,802AC9790,17425,80EC2E740,3174.84,US Dollar,3174.84,US Dollar,Cheque
5,02c4d40b869bd284,2022-09-01T00:00:00,2022-09-01,Thursday,00:00:00.000000,13264,803D0EB90,227171,80EC2DDC0,3198.62,US Dollar,3198.62,US Dollar,ACH
6,4f5d7717956140c7,2022-09-01T00:00:00,2022-09-01,Thursday,00:00:00.000000,45701,81130A2B0,45701,81130A2B0,6160.74,Shekel,6160.74,Shekel,Reinvestment
7,482dad2c990b5202,2022-09-01T00:00:00,2022-09-01,Thursday,00:00:00.000000,8354,807350F90,8354,807350F90,17.71,US Dollar,17.71,US Dollar,Reinvestment
8,32936ef0a9a79bc4,2022-09-01T00:00:00,2022-09-01,Thursday,00:00:00.000000,1132,80BE5E680,1132,80BE5E680,1057246.47,Mexican Peso,1057246.47,Mexican Peso,Reinvestment
9,7b310ee1aaf1d6c9,2022-09-01T00:00:00,2022-09-01,Thursday,00:00:00.000000,39900,803D91A80,210261,803D92040,0.03,Yen,0.03,Yen,Cheque


In [17]:
# Preview the account_transactions view
# Notice each transaction appears as both an IN row and an OUT row
sample_tx = pd.read_sql("SELECT transaction_id FROM transactions LIMIT 1", conn).iloc[0]["transaction_id"]

print(f"Looking up transaction: {sample_tx}\n")
pd.read_sql(f"SELECT * FROM account_transactions WHERE transaction_id = '{sample_tx}'", conn)

Looking up transaction: 000002908b4740a1



,transaction_id,timestamp,account,direction,counterparty,bank,counterparty_bank,amount,currency,payment_format
0,000002908b4740a1,2022-09-02T23:43:00,803A621E0,OUT,803F400C0,21,10099,290618.32,Yen,Cheque
1,000002908b4740a1,2022-09-02T23:43:00,803F400C0,IN,803A621E0,10099,21,290618.32,Yen,Cheque


In [18]:
conn.close()

## 4. The `ReadOnlySqlDatabase` Tool

So far we've been using `sqlite3` directly — a regular connection that could run `DROP TABLE` or `DELETE` if we wanted. 

When an LLM agent runs SQL, we can't have it modifying data. The [`ReadOnlySqlDatabase`](https://github.com/VectorInstitute/eval-agents/blob/main/aieng-eval-agents/aieng/agent_evals/tools/sql_database.py) tool solves this with two layers of protection:

1. **AST-level enforcement** — It parses the SQL into a syntax tree using [SQLGlot](https://sqlglot.com/) and rejects any query that contains write operations (`INSERT`, `UPDATE`, `DROP`, etc.), even if hidden inside a CTE or subquery.
2. **Row limits + timeouts** — It caps results at `max_rows` (default 100) and cancels slow queries, preventing runaway costs.

The tool exposes exactly **two methods** that become the agent's "tools":
- `get_schema_info()` — returns the table/column names
- `execute(query)` — runs a SELECT and returns a markdown table string

In [19]:
db = ReadOnlySqlDatabase(
    connection_uri=f"sqlite:///{DB_PATH}",
    agent_name="NotebookExplorer",
    max_rows=10,  # keep output short for this notebook
)

print("Tool created successfully!")
print(f"Max rows: {db.max_rows}")
print(f"Agent name: {db.agent_name}")

Tool created successfully!
Max rows: 10
Agent name: NotebookExplorer


### 4.1. Schema Discovery

This is the first thing the agent does on every case — ask "what tables exist and what columns do they have?"

In [20]:
schema = db.get_schema_info()
print(schema)

Table: accounts
  Columns: bank_name: TEXT, bank_id: INTEGER, account_number: TEXT, entity_id: TEXT, entity_name: TEXT
Table: transactions
  Columns: transaction_id: TEXT, timestamp: TEXT, date: TEXT, day_of_week: TEXT, time_of_day: TEXT, from_bank: INTEGER, from_account: TEXT, to_bank: INTEGER, to_account: TEXT, amount_received: REAL, receiving_currency: TEXT, amount_paid: REAL, payment_currency: TEXT, payment_format: TEXT
View: account_transactions
  Columns: transaction_id: TEXT, timestamp: TEXT, account: TEXT, direction: NULL, counterparty: TEXT, bank: INTEGER, counterparty_bank: INTEGER, amount: REAL, currency: TEXT, payment_format: TEXT


In [21]:
# You can also ask for a specific table only
print(db.get_schema_info(table_names=["transactions"]))

Table: transactions
  Columns: transaction_id: TEXT, timestamp: TEXT, date: TEXT, day_of_week: TEXT, time_of_day: TEXT, from_bank: INTEGER, from_account: TEXT, to_bank: INTEGER, to_account: TEXT, amount_received: REAL, receiving_currency: TEXT, amount_paid: REAL, payment_currency: TEXT, payment_format: TEXT


### 4.2. Running Queries Through the Tool

Notice that the output is a **markdown table string** — not a DataFrame. This is intentional: the agent receives it as plain text in its context window.

In [22]:
result = db.execute("SELECT * FROM accounts LIMIT 5")
print(result)

| bank_name | bank_id | account_number | entity_id | entity_name |
| --- | --- | --- | --- | --- |
| Portugal Bank #4507 | 331579 | 80B779D80 | 80062E240 | Sole Proprietorship #50438 |
| Canada Bank #27 | 210 | 809D86900 | 800C998A0 | Corporation #33520 |
| UK Bank #33 | 21884 | 80812BE00 | 800C47F50 | Partnership #35397 |
| Germany Bank #4815 | 32742 | 81047F300 | 80096F0B0 | Corporation #48813 |
| National Bank of Harrisburg | 127390 | 80BD8CF00 | 800FB8760 | Corporation #889 |


In [23]:
# Aggregation query. This is the kind of query the agent would run
result = db.execute("""
    SELECT
        account,
        COUNT(*) AS tx_count,
        COUNT(DISTINCT counterparty) AS unique_counterparties,
        SUM(CASE WHEN direction='IN' THEN amount ELSE 0 END) AS total_in,
        SUM(CASE WHEN direction='OUT' THEN amount ELSE 0 END) AS total_out
    FROM account_transactions
    GROUP BY account
    ORDER BY tx_count DESC
    LIMIT 10
""")
print(result)

| account | tx_count | unique_counterparties | total_in | total_out |
| --- | --- | --- | --- | --- |
| 100428660 | 169756 | 14775 | 479594.79 | 52762291209.75 |
| 1004286A8 | 103671 | 9174 | 251001.27 | 26069375477.37 |
| 100428978 | 20647 | 1851 | 44445.62 | 7532109611.71 |
| 1004286F0 | 18771 | 1630 | 253933.88 | 24657357382.93 |
| 100428780 | 17381 | 1482 | 10570889.95 | 373933699163.91003 |
| 1004289C0 | 16926 | 1500 | 313581.33 | 10517190140.53 |
| 100428810 | 16540 | 1452 | 27366.88 | 5153615843.1 |
| 1004287C8 | 14277 | 1236 | 5474418.39 | 101107241619.05 |
| 100428738 | 13854 | 1187 | 6304388.45 | 259216781715.29 |
| 100428A51 | 13101 | 1161 | 0.52359 | 262859.640408 |

... (Truncated at 10 rows) ...


### 4.3. Safety Demo — Write Operations Are Blocked

Let's verify the protection actually works. The tool should reject any write operation.

In [24]:
# Attempting a DELETE. This should be blocked
result = db.execute("DELETE FROM transactions WHERE 1=1")
print(result)  # Expect: "Query Error: Security Violation..."

2026-02-26 19:40:20,474 WARNING aieng.agent_evals.tools.sql_database: Blocked Unsafe Query Type: <class 'sqlglot.expressions.Delete'>


Query Error: Security Violation: Query contains prohibited WRITE operations.


In [25]:
# Attempting a write hidden inside a CTE — also blocked
result = db.execute("""
    WITH cleanup AS (
        DELETE FROM transactions WHERE 1=1
    )
    SELECT * FROM accounts LIMIT 1
""")
print(result)  # Expect: "Query Error: Security Violation..."

2026-02-26 19:40:20,538 WARNING aieng.agent_evals.tools.sql_database: CTE usage blocked by policy


Query Error: Security Violation: Query contains prohibited WRITE operations.


In [26]:
# Row limit enforcement — we set max_rows=10 above, so this won't return all rows
result = db.execute("SELECT * FROM transactions")
# Check the last line — it should say "Truncated at 10 rows"
for line in result.split("\n")[-3:]:
    print(line)

| 7b310ee1aaf1d6c9 | 2022-09-01T00:00:00 | 2022-09-01 | Thursday | 00:00:00.000000 | 39900 | 803D91A80 | 210261 | 803D92040 | 0.03 | Yen | 0.03 | Yen | Cheque |

... (Truncated at 10 rows) ...


## Summary

In this notebook you've seen:

1. **The dataset** — the IBM Transactions for Anti Money Laundering (AML) dataset, its structure, and how to download and clean it.
2. **The database** — two tables (`accounts`, `transactions`) and a convenience view (`account_transactions`) storing synthetic AML transaction data.
3. **Manual exploration** — how to use `pandas` + `sqlite3` to query the data as a developer would.
4. **The `ReadOnlySqlDatabase` tool** — the safety-hardened wrapper the agent uses, with AST-level write blocking and row limits.

**Next:** In Notebook 2, we'll instantiate the AML agent, explore how to give it tasks, then inspect its reasoning and tool call trace.

In [ ]:
db.close()
print("Done!")

Done!


: 